In [1]:
import pandas as pd
import numpy as np

In [2]:
fndds_all = pd.read_csv('../fndds_all_ingredient_nutrient_values.csv')
fndds_1920 = pd.read_csv('./fndds_ingredient_nutrient_1920.csv')
fndds_2123 = pd.read_csv('./fndds_ingredient_nutrient_2123.csv')

#### FDA-FDD has a single ingredient description to describe oils in mixed meals, to appropriately match this generic description, we will take the Vegetable oil, NFS food code and generate an ingredient (Vegetable oil, NFS, code:4321) that is the weighted average of oils consumed in the US:

In [3]:
# select the ingredient codes that represent vegetable oil, NFS:
veg_oil_nfs = fndds_2123.loc[fndds_2123['Ingredient code'].isin([4044, 4518, 4582, 100262, 4063, 100258])].copy()

# ingredient weights for the Vegetable oil, NFS:
weights = [0.45, 0.1, 0.15, 0.05, 0.175, 0.075]
codes = [4044, 4518, 4582, 100262, 4063, 100258]

# Create a mapping dictionary
weight_mapping = dict(zip(codes, weights))

# Add the weight column to your dataframe
veg_oil_nfs['weight'] = veg_oil_nfs['Ingredient code'].map(weight_mapping)

# Calculate weighted average for each nutrient group
veg_oil_nfs['weighted_avg_nutrients'] = veg_oil_nfs.groupby(['Nutrient description']).apply(
    lambda x: np.average(x['Nutrient value'], weights=x['weight'])
).reindex(veg_oil_nfs['Nutrient description']).values

#clean up dataframe to place back into fndds_2123 codes:
veg_oil_nfs = veg_oil_nfs.drop_duplicates(subset='Nutrient description')
veg_oil_nfs.drop(columns={'Nutrient value'}, inplace=True)
veg_oil_nfs.rename(columns={'weighted_avg_nutrients':'Nutrient value'}, inplace=True)
veg_oil_nfs['Ingredient code'] = 4321
veg_oil_nfs['Ingredient description'] = 'Vegetable oil, NFS'
veg_oil_nfs.drop(columns='weight', inplace=True)

# add back to ingredient ccodes
fndds_2123 = pd.concat([fndds_2123, veg_oil_nfs])

In [4]:
fndds_1920 = fndds_1920.pivot(columns = 'Nutrient description', values= ['Nutrient value'], index=['Ingredient code', 'Ingredient description'])

In [5]:
fndds_1920.columns = fndds_1920.columns.to_flat_index()
fndds_1920 = fndds_1920.reset_index()

In [6]:
fndds_1920.columns = fndds_all.columns

In [7]:
fndds_2123 = fndds_2123.pivot(columns = 'Nutrient description', values= ['Nutrient value'], index=['Ingredient code', 'Ingredient description'])

In [8]:
fndds_2123.columns = fndds_2123.columns.to_flat_index()
fndds_2123 = fndds_2123.reset_index()

In [9]:
fndds_2123.columns = fndds_all.columns

In [10]:
add_1920 = fndds_1920[~fndds_1920['Ingredient code'].isin(fndds_all['Ingredient code'])]

In [11]:
add_2123 = fndds_2123[~fndds_2123['Ingredient code'].isin(fndds_all['Ingredient code'])]

In [12]:
add_2123 = add_2123[~add_2123['Ingredient code'].isin(add_1920['Ingredient code'])]

In [13]:
fndds_all

,Ingredient code,Ingredient description,Capric acid,Lauric acid,Myristic acid,Palmitic acid,Palmitoleic acid,Stearic acid,Oleic acid,Linoleic acid,...,Vitamin B-12,"Vitamin B-12, added",Vitamin B-6,Vitamin C,Vitamin D (D2 + D3),Vitamin E (alpha-tocopherol),"Vitamin E, added",Vitamin K (phylloquinone),Water,Zinc
0,1001,"Butter, salted",2.529,2.587,7.436,21.697,0.961,9.999,19.961,2.728,...,0.17,0.0,0.003,0.0,0.0,2.32,0.0,7.0,15.87,0.09
1,1002,"Butter, whipped, with salt",2.039,2.354,7.515,20.531,1.417,7.649,17.370,2.713,...,0.07,0.0,0.008,0.0,0.0,1.37,0.0,4.6,16.72,0.05
2,1003,"Butter oil, anhydrous",2.495,2.793,10.005,26.166,2.228,12.056,25.026,2.247,...,0.01,0.0,0.001,0.0,0.0,2.80,0.0,8.6,0.24,0.01
3,1004,"Cheese, blue",0.601,0.491,3.301,9.153,0.816,3.235,6.622,0.536,...,1.22,0.0,0.166,0.0,0.5,0.25,0.0,2.4,42.41,2.66
4,1005,"Cheese, brick",0.585,0.482,3.227,8.655,0.817,3.455,7.401,0.491,...,1.26,0.0,0.065,0.0,0.5,0.26,0.0,2.5,41.11,2.60
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2739,19312,"Pie fillings, apple, canned",0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,...,0.00,0.0,0.016,1.7,0.0,0.04,0.0,0.5,73.40,0.04
2740,21610,"School Lunch, chicken patty, whole grain breaded",0.000,0.000,0.000,2.162,0.361,0.481,4.035,6.419,...,0.31,0.0,0.195,0.7,0.3,1.36,0.0,4.9,53.84,1.12
2741,23090,"Beef, chuck for stew, separable lean and fat, ...",0.000,0.000,0.173,1.546,0.238,0.989,3.146,0.366,...,2.73,0.0,0.511,0.0,0.1,0.11,0.0,1.6,60.08,8.32
2742,43449,"Beans, baked, canned, no salt added",0.000,0.000,0.000,0.097,0.000,0.006,0.035,0.093,...,0.00,0.0,0.130,3.1,0.0,0.15,0.0,0.8,72.60,1.40


In [ ]:
pd.concat([fndds_all, add_1920, add_2123], axis=0).drop_duplicates(subset='Ingredient code').to_csv('../ingredient_matching/fndds_all_ingredient_nutrient_values_2023.csv', index=None)